# ASR Grouping Analysis

End-to-end look at the grouping step produced by `scripts/dataset/group_and_sample_asr_manifest.py`:

1. **Part 1 — Manifest segment-length analysis.** Inspects the five grouped manifests (`asr_manifest_Ns_20k.jsonl` for `N ∈ {2, 3, 5, 10, 15}`) plus the ungrouped `asr_manifest.jsonl` baseline. Focus is segment duration, word count, and grouping depth (`n_merged`).
2. **Part 2 — ASR prediction analysis.** Evaluates a single model (`whisperx:distil-large-v3`) run against each of the five grouped manifests, to see how the grouping threshold affects WER, latency, and error composition.

# Part 1 — Manifest Segment-Length Analysis

The manifests in `data/processed/manifests/asr_manifest_*s_20k.jsonl` are 20k-record samples produced by `scripts/dataset/group_and_sample_asr_manifest.py`, each generated with a different `MIN_DURATION` threshold (2s, 3s, 5s, 10s, 15s). The ungrouped `asr_manifest.jsonl` is included as a baseline to show the pre-grouping distribution.

The grouping script merges consecutive utterances from the same speaker until they reach `MIN_DURATION`, capped at `MAX_DURATION` (30s in the actual runs) and broken by `GAP_THRESHOLD=3s` gaps. This section inspects how segment length (duration, word count, `n_merged`) distributes across the manifests so we can pick sensible thresholds for downstream ASR/NER evaluation.

Note: the `original` manifest is much larger than 20k rows, so histograms use density rather than raw counts so the shapes remain comparable.

## Configuration

In [ ]:
from pathlib import Path
from collections import Counter
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from jiwer import process_words, process_characters, wer

In [ ]:
# ===== CONFIG =====
MANIFESTS_DIR = Path("../data/processed/manifests/")

# Each entry is (label, filename). Label is used as the display name throughout.
# "original" is the ungrouped manifest — baseline showing the pre-grouping distribution.
# The other labels' min_dur matches the MIN_DURATION used when generating that manifest.
MANIFEST_FILES = [
    ("original", "asr_manifest.jsonl"),
    ("2s",       "asr_manifest_2s_20k.jsonl"),
    ("3s",       "asr_manifest_3s_20k.jsonl"),
    ("5s",       "asr_manifest_5s_20k.jsonl"),
    ("10s",      "asr_manifest_10s_20k.jsonl"),
    ("15s",      "asr_manifest_15s_20k.jsonl"),
]

# Duration buckets (seconds) used for stratified counts.
DURATION_BUCKETS = [0, 2, 5, 10, 15, 20, 25]

# Word-count buckets for reference_text length.
WORD_BUCKETS = [0, 5, 10, 20, 40, 80, float("inf")]

## Data Loading

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def load_manifests(folder, files):
    frames = []
    for label, filename in files:
        rows = load_jsonl(Path(folder) / filename)
        df = pd.DataFrame(rows)
        df["manifest"] = label
        if "duration" not in df.columns:
            df["duration"] = df["end_time"] - df["start_time"]
        df["duration"] = df["duration"].astype(float)
        df["word_count"] = df["reference_text"].fillna("").apply(lambda t: len(t.split()))
        df["char_count"] = df["reference_text"].fillna("").str.len()
        if "n_merged" not in df.columns:
            df["n_merged"] = 1
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

manifests = load_manifests(MANIFESTS_DIR, MANIFEST_FILES)
manifest_order = [lbl for lbl, _ in MANIFEST_FILES]
manifests["manifest"] = pd.Categorical(manifests["manifest"], categories=manifest_order, ordered=True)
print(f"Loaded {len(manifests)} rows across {manifests['manifest'].nunique()} manifests")
manifests.head()

## Summary Statistics

In [ ]:
# =========================
# Per-manifest summary: duration, word count, and grouping stats.
# These should shift predictably with MIN_DURATION — higher MIN_DURATION
# means longer segments and more merged utterances per record.
# =========================
def summarise(group):
    d = group["duration"]
    w = group["word_count"]
    m = group["n_merged"]
    return pd.Series({
        "n_records":       len(group),
        "dur_mean":        d.mean(),
        "dur_median":      d.median(),
        "dur_p90":         d.quantile(0.90),
        "dur_min":         d.min(),
        "dur_max":         d.max(),
        "total_hours":     d.sum() / 3600.0,
        "words_mean":      w.mean(),
        "words_median":    w.median(),
        "n_merged_mean":   m.mean(),
        "n_merged_max":    m.max(),
        "pct_single_seg":  (m == 1).mean() * 100,
        "pct_overlap":     group["overlap"].mean() * 100,
        "pct_empty_text":  (group["word_count"] == 0).mean() * 100,
    })

summary_manifest = manifests.groupby("manifest", observed=True).apply(summarise).round(2)
display(summary_manifest)

### Duration Distribution

In [ ]:
# =========================
# Duration bucket counts per manifest.
# Shows how the MIN_DURATION threshold shifts mass out of short buckets.
# =========================
labels = [f"{DURATION_BUCKETS[i]}-{DURATION_BUCKETS[i+1]}s" for i in range(len(DURATION_BUCKETS)-1)]
labels.append(f">{DURATION_BUCKETS[-1]}s")
edges = DURATION_BUCKETS + [float("inf")]

manifests["dur_bucket"] = pd.cut(manifests["duration"], bins=edges, labels=labels, right=False, include_lowest=True)

bucket_table = (
    manifests.groupby(["manifest", "dur_bucket"], observed=True)
      .size()
      .unstack("dur_bucket", fill_value=0)
)
bucket_pct = bucket_table.div(bucket_table.sum(axis=1), axis=0) * 100
print("Counts per duration bucket:")
display(bucket_table)
print("\nPercentage per duration bucket:")
display(bucket_pct.round(1))

In [ ]:
# =========================
# Overlaid duration histograms (density — manifests have different sizes).
# =========================
fig, ax = plt.subplots(figsize=(10, 5))
bins = np.arange(0, 31, 0.5)
for label in manifest_order:
    subset = manifests[manifests["manifest"] == label]["duration"].clip(upper=30)
    ax.hist(subset, bins=bins, alpha=0.4, label=label, histtype="stepfilled", density=True)
ax.set_xlabel("Segment duration (s)")
ax.set_ylabel("Density")
ax.set_title("Duration distribution per manifest")
ax.legend(title="Manifest")
ax.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Box plot — duration spread per manifest.
# =========================
fig, ax = plt.subplots(figsize=(9, 5))
data = [manifests[manifests["manifest"] == label]["duration"].values for label in manifest_order]
bp = ax.boxplot(data, labels=manifest_order, patch_artist=True,
                showfliers=True, flierprops=dict(marker=".", markersize=3, alpha=0.3))
for patch in bp["boxes"]:
    patch.set_facecolor("#d0e8f7")
ax.set_xlabel("Manifest (MIN_DURATION)")
ax.set_ylabel("Segment duration (s)")
ax.set_title("Duration distribution per manifest")
ax.grid(True, axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

### Word-Count Distribution

In [ ]:
# =========================
# Word-count bucket counts per manifest.
# Short utterances (<5 words) are the ones that inflate WER.
# =========================
word_labels = []
for i in range(len(WORD_BUCKETS)-1):
    hi = WORD_BUCKETS[i+1]
    lo = WORD_BUCKETS[i]
    if hi == float("inf"):
        word_labels.append(f">={int(lo)}")
    else:
        word_labels.append(f"{int(lo)}-{int(hi)-1}")

manifests["word_bucket"] = pd.cut(manifests["word_count"], bins=WORD_BUCKETS, labels=word_labels, right=False, include_lowest=True)

word_table = (
    manifests.groupby(["manifest", "word_bucket"], observed=True)
      .size()
      .unstack("word_bucket", fill_value=0)
)
word_pct = word_table.div(word_table.sum(axis=1), axis=0) * 100
print("Counts per word-count bucket:")
display(word_table)
print("\nPercentage per word-count bucket:")
display(word_pct.round(1))

In [ ]:
# =========================
# Duration vs word count scatter, one panel per manifest.
# =========================
fig, axes = plt.subplots(1, len(manifest_order), figsize=(3.5 * len(manifest_order), 4), sharey=True)
if len(manifest_order) == 1:
    axes = [axes]
for ax, label in zip(axes, manifest_order):
    sub = manifests[manifests["manifest"] == label]
    ax.scatter(sub["duration"], sub["word_count"], s=4, alpha=0.25, color="#5b8db8")
    ax.set_title(f"MIN_DURATION = {label}")
    ax.set_xlabel("Duration (s)")
    ax.grid(True, linestyle="--", alpha=0.3)
axes[0].set_ylabel("Word count")
plt.tight_layout()
plt.show()

### Grouping Depth (`n_merged`)

In [ ]:
# =========================
# How many original utterances are packed into each grouped record.
# The "original" manifest is by definition all n_merged=1.
# =========================
merged_table = (
    manifests.groupby(["manifest", "n_merged"], observed=True)
      .size()
      .unstack("n_merged", fill_value=0)
)
print("Records per n_merged:")
display(merged_table)

fig, ax = plt.subplots(figsize=(10, 5))
max_merged = int(manifests["n_merged"].max())
bins = np.arange(0.5, max_merged + 1.5, 1)
for label in manifest_order:
    subset = manifests[manifests["manifest"] == label]["n_merged"]
    ax.hist(subset, bins=bins, alpha=0.4, label=label, histtype="stepfilled", density=True)
ax.set_xlabel("n_merged (original utterances per record)")
ax.set_ylabel("Density")
ax.set_title("Grouping depth per manifest")
ax.set_yscale("log")
ax.legend(title="Manifest")
ax.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

### Overlap and Empty-Text Shares

In [ ]:
# =========================
# Two side effects of aggressive grouping:
#   - A group is "overlap" if ANY constituent utterance overlapped with another speaker.
#   - Empty reference_text can appear when the grouped utterances are all silence/noise.
# =========================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

overlap_pct = summary_manifest["pct_overlap"]
ax1.bar(overlap_pct.index.astype(str), overlap_pct.values, color="#f4a460")
ax1.set_ylabel("% of records flagged overlap")
ax1.set_title("Overlap share per manifest")
ax1.grid(True, axis="y", linestyle="--", alpha=0.4)
for i, v in enumerate(overlap_pct.values):
    ax1.text(i, v, f"{v:.1f}%", ha="center", va="bottom", fontsize=9)

empty_pct = summary_manifest["pct_empty_text"]
ax2.bar(empty_pct.index.astype(str), empty_pct.values, color="#8fbc8f")
ax2.set_ylabel("% of records with empty reference_text")
ax2.set_title("Empty-text share per manifest")
ax2.grid(True, axis="y", linestyle="--", alpha=0.4)
for i, v in enumerate(empty_pct.values):
    ax2.text(i, v, f"{v:.2f}%", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

### Takeaways

Use the tables and plots above to pick a manifest for downstream ASR/NER benchmarking. The tradeoff at higher `MIN_DURATION`:

- **Pros**: more context per ASR call, fewer 1-3 word segments (which dominate WER), closer to Whisper's 10-30s sweet spot.
- **Cons**: higher overlap share (harder audio), longer inference per segment, less granular alignment for NER error propagation.

# Part 2 — Grouped ASR Prediction Analysis

All predictions under `data/processed/asr_predictions/grouped/` were produced by the same model (`whisperx:distil-large-v3`) evaluated against the five grouped manifests. The axis of variation here is the manifest (i.e. the `MIN_DURATION` threshold used when grouping), so this section asks: how does grouping affect WER, latency, and error composition for a fixed model?

Because each manifest is a different sample, we cannot intersect on common segments (as in `asr_analysis.ipynb`). Instead, each manifest is evaluated on its own segment set and compared via aggregate metrics and length-stratified breakdowns.

## Configuration

In [ ]:
# ===== CONFIG =====
PREDICTIONS_DIR = Path("../data/processed/asr_predictions/grouped/")

# (manifest_label, filename). All share the same model — varies only by grouping threshold.
PREDICTION_FILES = [
    ("2s",  "whisperx_distil-large-v3_2s_predictions.jsonl"),
    ("3s",  "whisperx_distil-large-v3_3s_predictions.jsonl"),
    ("5s",  "whisperx_distil-large-v3_5s_predictions.jsonl"),
    ("10s", "whisperx_distil-large-v3_10s_predictions.jsonl"),
    ("15s", "whisperx_distil-large-v3_15s_predictions.jsonl"),
]
pred_manifest_order = [lbl for lbl, _ in PREDICTION_FILES]

# Filler words stripped in the 'filtered' metric variant.
FILLER_WORDS = {
    "uh", "um", "mm", "hmm", "hm", "mhm", "mmhm", "mmhmm",
    "ah", "eh", "er", "erm", "huh", "uhu", "uhuh", "mmm"
}

MIN_REF_CHARS = 0
WARMUP_DROP = 5
OUTLIER_PERCENTILE = 98
METRICS = ["wer", "mer", "wil"]

In [ ]:
def load_jsonl_df(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

def load_all_predictions(folder, files):
    frames = []
    for label, filename in files:
        d = load_jsonl_df(Path(folder) / filename)
        d["manifest"] = label
        frames.append(d)
    return pd.concat(frames, ignore_index=True)

def filter_fillers(text):
    text = str(text).lower()
    text = text.replace("-", " ").replace("_", " ")
    text = re.sub(r"[^\w\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return " ".join(w for w in text.split() if w not in FILLER_WORDS)

## Data Loading

In [ ]:
df_raw = load_all_predictions(PREDICTIONS_DIR, PREDICTION_FILES)
df_raw["manifest"] = pd.Categorical(df_raw["manifest"], categories=pred_manifest_order, ordered=True)
print(f"Loaded {len(df_raw)} rows across {df_raw['manifest'].nunique()} manifests")
df_raw.groupby("manifest", observed=True).size().rename("n_rows").to_frame()

## Preprocessing

In [ ]:
# =========================
# Each manifest is a different sample, so we don't intersect segments.
# We do apply MIN_REF_CHARS and drop rows where ref and hyp collapse to empty.
# =========================
df = df_raw.copy()
df["ref_num_chars"] = df["reference_text"].astype(str).str.len()

if MIN_REF_CHARS > 0:
    df = df[df["ref_num_chars"] >= MIN_REF_CHARS].copy()

df["ref_raw"]      = df["reference_text"].astype(str)
df["hyp_raw"]      = df["asr_text"].astype(str)
df["ref_filtered"] = df["reference_text"].apply(filter_fillers)
df["hyp_filtered"] = df["asr_text"].apply(filter_fillers)

df_eval = df[~((df["ref_filtered"] == "") & (df["hyp_filtered"] == ""))].copy()
print(f"Rows after filler-empty removal: {len(df_eval)} (from {len(df)})")

In [ ]:
# =========================
# LATENCY — trim warm-up and clip top percentile per manifest
# =========================
def trim_latencies(df, warmup_drop, outlier_pct):
    pieces = []
    for manifest, group in df.groupby("manifest", observed=True):
        g = group.iloc[warmup_drop:].copy()
        threshold = g["latency_ms"].quantile(outlier_pct / 100)
        pieces.append(g[g["latency_ms"] <= threshold])
    return pd.concat(pieces, ignore_index=True)

df_latency = trim_latencies(df_eval, WARMUP_DROP, OUTLIER_PERCENTILE)
n_dropped = len(df_eval) - len(df_latency)
print(f"Latency: dropped {n_dropped} rows ({n_dropped / len(df_eval) * 100:.1f}%) — "
      f"{WARMUP_DROP} warmup + top {100 - OUTLIER_PERCENTILE}% outliers per manifest")

latency_df = (
    df_latency.groupby("manifest", observed=True)["latency_ms"]
    .agg(latency_mean="mean", latency_p50="median",
         latency_p95=lambda x: x.quantile(0.95))
    .round(2)
    .reindex(pred_manifest_order)
)
print("\nLatency summary (ms):")
display(latency_df)

## Evaluation

In [ ]:
# =========================
# METRIC HELPERS
# Single groupby pass: raw + filtered metrics computed together.
# =========================
_WORD_METRICS = {"wer", "mer", "wil"}

def compute_all_metrics(group):
    need_word = bool(set(METRICS) & _WORD_METRICS)
    need_char = "cer" in METRICS

    result = {
        "num_segments":   len(group),
        "mean_ref_chars": group["ref_num_chars"].mean(),
        "mean_duration":  group["duration"].mean(),
    }

    for variant, ref_col, hyp_col in [
        ("raw",      "ref_raw",      "hyp_raw"),
        ("filtered", "ref_filtered", "hyp_filtered"),
    ]:
        if need_word:
            w = process_words(group[ref_col].tolist(), group[hyp_col].tolist())
            if "wer" in METRICS: result[f"{variant}_wer"] = w.wer
            if "mer" in METRICS: result[f"{variant}_mer"] = w.mer
            if "wil" in METRICS: result[f"{variant}_wil"] = w.wil
        if need_char:
            c = process_characters(" ".join(group[ref_col]), " ".join(group[hyp_col]))
            result[f"{variant}_cer"] = c.cer

    return pd.Series(result)

### Tables

In [ ]:
summary = (
    df_eval.groupby("manifest", observed=True)
    .apply(compute_all_metrics)
    .reindex(pred_manifest_order)
).join(latency_df)
display(summary.round(4))

In [ ]:
raw_cols = {f"raw_{m}": m.upper() for m in METRICS if f"raw_{m}" in summary.columns}
raw_table = summary.reset_index()[["manifest"] + list(raw_cols) + ["latency_mean"]].rename(
    columns={"manifest": "Manifest", "latency_mean": "Latency (ms)", **raw_cols}
)
for col in raw_cols.values():
    raw_table[col] = raw_table[col].round(4)
raw_table["Latency (ms)"] = raw_table["Latency (ms)"].round(1)
print("Raw metrics (no filler filter):")
display(raw_table)

In [ ]:
filt_cols = {f"filtered_{m}": m.upper() for m in METRICS if f"filtered_{m}" in summary.columns}
filt_table = summary.reset_index()[["manifest"] + list(filt_cols) + ["latency_mean"]].rename(
    columns={"manifest": "Manifest", "latency_mean": "Latency (ms)", **filt_cols}
)
for col in filt_cols.values():
    filt_table[col] = filt_table[col].round(4)
filt_table["Latency (ms)"] = filt_table["Latency (ms)"].round(1)
print("Filtered metrics (filler words removed):")
display(filt_table)

### No-Overlap Evaluation

Overlap flags propagate upward during grouping (a group is overlap if any constituent utterance was). Expect a smaller non-overlapping subset at higher `MIN_DURATION` and a cleaner WER ceiling.

In [ ]:
df_no_overlap = df_eval[df_eval["overlap"] == False].copy()
print("Non-overlapping segments per manifest:")
display(df_no_overlap.groupby("manifest", observed=True).size().rename("n_rows").to_frame())

summary_no = (
    df_no_overlap.groupby("manifest", observed=True)
    .apply(compute_all_metrics)
    .reindex(pred_manifest_order)
).join(latency_df)

raw_no = summary_no.reset_index()[["manifest"] + list(raw_cols) + ["latency_mean"]].rename(
    columns={"manifest": "Manifest", "latency_mean": "Latency (ms)", **raw_cols}
)
for col in raw_cols.values():
    raw_no[col] = raw_no[col].round(4)
raw_no["Latency (ms)"] = raw_no["Latency (ms)"].round(1)
print("\nRaw metrics — non-overlapping only:")
display(raw_no)

filt_no = summary_no.reset_index()[["manifest"] + list(filt_cols) + ["latency_mean"]].rename(
    columns={"manifest": "Manifest", "latency_mean": "Latency (ms)", **filt_cols}
)
for col in filt_cols.values():
    filt_no[col] = filt_no[col].round(4)
filt_no["Latency (ms)"] = filt_no["Latency (ms)"].round(1)
print("\nFiltered metrics — non-overlapping only:")
display(filt_no)

### Plots

In [ ]:
# =========================
# Filtered WER and latency per manifest (dual y-axis bar + line).
# =========================
fig, ax1 = plt.subplots(figsize=(9, 5))
x = np.arange(len(pred_manifest_order))

ax1.bar(x, summary["filtered_wer"].values, color="#5b8db8", alpha=0.8, label="Filtered WER")
ax1.set_xticks(x)
ax1.set_xticklabels(pred_manifest_order)
ax1.set_xlabel("Manifest (MIN_DURATION)")
ax1.set_ylabel("Filtered WER", color="#5b8db8")
ax1.tick_params(axis="y", labelcolor="#5b8db8")
for xi, v in zip(x, summary["filtered_wer"].values):
    ax1.text(xi, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9, color="#3a5d7a")

ax2 = ax1.twinx()
ax2.plot(x, summary["latency_mean"].values, color="#c44e52", marker="o", label="Mean latency")
ax2.set_ylabel("Mean latency (ms)", color="#c44e52")
ax2.tick_params(axis="y", labelcolor="#c44e52")

ax1.set_title("Filtered WER and mean latency per manifest")
ax1.grid(True, axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Latency distribution box plot per manifest.
# =========================
fig, ax = plt.subplots(figsize=(10, 5))
data = [df_latency[df_latency["manifest"] == m]["latency_ms"].values for m in pred_manifest_order]
bp = ax.boxplot(data, labels=pred_manifest_order, patch_artist=True,
                showfliers=True, flierprops=dict(marker=".", markersize=3, alpha=0.3))
for patch in bp["boxes"]:
    patch.set_facecolor("#d0e8f7")
ax.set_xlabel("Manifest (MIN_DURATION)")
ax.set_ylabel("Latency per segment (ms)")
ax.set_title("Latency distribution per manifest")
ax.grid(True, axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Filtered WER/MER/WIL grouped bar per manifest
# =========================
metric_names = [m.upper() for m in METRICS if f"filtered_{m}" in summary.columns]
values = np.array([[summary.loc[m, f"filtered_{met.lower()}"] for met in metric_names]
                   for m in pred_manifest_order])

x = np.arange(len(pred_manifest_order))
width = 0.8 / len(metric_names)
colors = ["#5b8db8", "#f4a460", "#8fbc8f"]

fig, ax = plt.subplots(figsize=(9, 5))
for i, name in enumerate(metric_names):
    ax.bar(x + i * width - (len(metric_names)-1) * width / 2, values[:, i],
           width=width, label=name, color=colors[i % len(colors)])
ax.set_xticks(x)
ax.set_xticklabels(pred_manifest_order)
ax.set_xlabel("Manifest (MIN_DURATION)")
ax.set_ylabel("Error rate (filtered)")
ax.set_title("Filtered WER / MER / WIL per manifest")
ax.legend()
ax.grid(True, axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Latency vs segment duration per manifest.
# =========================
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0, 0.9, len(pred_manifest_order)))
for color, m in zip(colors, pred_manifest_order):
    sub = df_latency[df_latency["manifest"] == m]
    ax.scatter(sub["duration"], sub["latency_ms"], s=6, alpha=0.25, color=color, label=m)
ax.set_xlabel("Segment duration (s)")
ax.set_ylabel("Latency (ms)")
ax.set_title("Latency vs segment duration per manifest")
ax.legend(title="MIN_DURATION", markerscale=3)
ax.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Substitution / Deletion / Insertion breakdown per manifest.
#   Substitutions -> wrong words (NER: wrong entity text/label)
#   Deletions     -> dropped words (NER: false negatives)
#   Insertions    -> hallucinated words (NER: false positives)
# =========================
rows = []
for m in pred_manifest_order:
    grp = df_eval[df_eval["manifest"] == m]
    out = process_words(grp["ref_filtered"].tolist(), grp["hyp_filtered"].tolist())
    total = out.substitutions + out.deletions + out.insertions
    if total == 0:
        continue
    rows.append({
        "manifest":      m,
        "Substitutions": out.substitutions / total,
        "Deletions":     out.deletions     / total,
        "Insertions":    out.insertions    / total,
    })

err_df = pd.DataFrame(rows).set_index("manifest")
ax = err_df.plot(kind="barh", stacked=True, figsize=(10, 4.5),
                 color=["#5b8db8", "#f4a460", "#8fbc8f"])
ax.set_xlabel("Proportion of total errors")
ax.set_ylabel("Manifest")
ax.set_title("Error type breakdown per manifest (filtered text)")
ax.legend(loc="lower right")
ax.grid(True, axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

### Investigations

In [ ]:
# =========================
# INVESTIGATION 1 — WER by reference word-count bucket, per manifest.
# Short utterances are where Whisper tends to hallucinate context.
# =========================
length_buckets = [
    (1,   3,  "1-3 words"),
    (4,   6,  "4-6 words"),
    (7,   15, "7-15 words"),
    (16,  30, "16-30 words"),
    (31, 999, ">30 words"),
]
bucket_order = [b[2] for b in length_buckets]

length_rows = []
for lo, hi, label in length_buckets:
    mask = df_eval["ref_filtered"].apply(lambda t: lo <= len(t.split()) <= hi)
    subset = df_eval[mask]
    for m, grp in subset.groupby("manifest", observed=True):
        if grp.empty:
            continue
        w = process_words(grp["ref_filtered"].tolist(), grp["hyp_filtered"].tolist())
        length_rows.append({
            "Length bucket": label,
            "manifest":      m,
            "Segments":      len(grp),
            "WER":           round(w.wer, 4),
        })
length_df = pd.DataFrame(length_rows)

fig, ax = plt.subplots(figsize=(10, 5))
for m in pred_manifest_order:
    grp = length_df[length_df["manifest"] == m].set_index("Length bucket").reindex(bucket_order)
    ax.plot(grp.index, grp["WER"], marker="o", label=m)
ax.set_xlabel("Reference length bucket")
ax.set_ylabel("Filtered WER")
ax.set_title("WER by segment length per manifest")
ax.legend(title="MIN_DURATION", fontsize=8)
ax.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

print("\nSegment counts per bucket and manifest:")
display(
    length_df.pivot(index="Length bucket", columns="manifest", values="Segments")
             .reindex(bucket_order)[pred_manifest_order]
)

In [ ]:
# =========================
# INVESTIGATION 2 — WER for overlap vs no-overlap, per manifest.
# =========================
overlap_rows = []
for m, grp in df_eval.groupby("manifest", observed=True):
    for flag, label in [(True, "Overlap"), (False, "No overlap")]:
        subset = grp[grp["overlap"] == flag]
        if subset.empty:
            continue
        w = process_words(subset["ref_filtered"].tolist(), subset["hyp_filtered"].tolist())
        overlap_rows.append({
            "manifest":  m,
            "Condition": label,
            "Segments":  len(subset),
            "WER":       round(w.wer, 4),
        })
overlap_df = pd.DataFrame(overlap_rows)

pivot = overlap_df.pivot(index="manifest", columns="Condition", values="WER").reindex(pred_manifest_order)
ax = pivot.plot(kind="bar", figsize=(9, 5), color=["#5b8db8", "#f4a460"])
ax.set_ylabel("Filtered WER")
ax.set_title("WER: overlap vs non-overlap per manifest")
ax.set_xlabel("Manifest (MIN_DURATION)")
ax.legend(title="")
ax.grid(True, axis="y", linestyle="--", alpha=0.4)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("\nSegment counts:")
display(
    overlap_df.pivot(index="manifest", columns="Condition", values="Segments").reindex(pred_manifest_order)
)

### Error Diagnostics

Top deletions, insertions, and substitution pairs per manifest. Filtered text variant is used so filler words don't swamp the lists.

In [ ]:
TOP_N = 25

def collect_errors(refs, hyps):
    out = process_words(refs, hyps)
    deletions, insertions, substitutions = Counter(), Counter(), Counter()
    for r_sent, h_sent, chunks in zip(out.references, out.hypotheses, out.alignments):
        for c in chunks:
            if c.type == "delete":
                for w in r_sent[c.ref_start_idx:c.ref_end_idx]:
                    deletions[w] += 1
            elif c.type == "insert":
                for w in h_sent[c.hyp_start_idx:c.hyp_end_idx]:
                    insertions[w] += 1
            elif c.type == "substitute":
                r_tok = r_sent[c.ref_start_idx:c.ref_end_idx]
                h_tok = h_sent[c.hyp_start_idx:c.hyp_end_idx]
                for a, b in zip(r_tok, h_tok):
                    substitutions[(a, b)] += 1
    return deletions, insertions, substitutions


error_stats = {}
for m, grp in df_eval.groupby("manifest", observed=True):
    error_stats[m] = dict(zip(
        ["deletions", "insertions", "substitutions"],
        collect_errors(grp["ref_filtered"].tolist(), grp["hyp_filtered"].tolist()),
    ))

all_del = Counter()
for s in error_stats.values():
    all_del.update(s["deletions"])
top_del = [w for w, _ in all_del.most_common(TOP_N)]
del_tbl = pd.DataFrame(
    {m: [error_stats[m]["deletions"].get(w, 0) for w in top_del] for m in pred_manifest_order},
    index=top_del,
)
del_tbl["TOTAL"] = del_tbl.sum(axis=1)
del_tbl = del_tbl.sort_values("TOTAL", ascending=False)
print(f"Top {TOP_N} deleted reference words:")
display(del_tbl)

all_ins = Counter()
for s in error_stats.values():
    all_ins.update(s["insertions"])
top_ins = [w for w, _ in all_ins.most_common(TOP_N)]
ins_tbl = pd.DataFrame(
    {m: [error_stats[m]["insertions"].get(w, 0) for w in top_ins] for m in pred_manifest_order},
    index=top_ins,
)
ins_tbl["TOTAL"] = ins_tbl.sum(axis=1)
ins_tbl = ins_tbl.sort_values("TOTAL", ascending=False)
print(f"\nTop {TOP_N} inserted words (possible hallucinations):")
display(ins_tbl)

all_sub = Counter()
for s in error_stats.values():
    all_sub.update(s["substitutions"])
sub_rows = []
for (r, h), total in all_sub.most_common(TOP_N):
    row = {"ref": r, "hyp": h, "TOTAL": total}
    for m in pred_manifest_order:
        row[m] = error_stats[m]["substitutions"].get((r, h), 0)
    sub_rows.append(row)
print(f"\nTop {TOP_N} substitution pairs (ref -> hyp):")
display(pd.DataFrame(sub_rows))